In [ ]:
import pandas as pd
import openai
from time import time , sleep
import json
import openai

df=pd.read_csv(r'test_data.csv')

In [ ]:
import re
def remove_urls(text):
    # Define regex pattern to match URLs starting with http:// or https://
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

    # Replace URLs with an empty string
    text_without_urls = re.sub(url_pattern, '', text)

    return text_without_urls

df['Link_removed']=df['tweet'].apply(remove_urls)

In [ ]:
df=df.iloc[445:]

In [ ]:
len(df)

In [ ]:
# install open ai library
!pip install openai

In [ ]:
openai.api_key = 'Your_Key'

In [ ]:
prompt_template = """
You are an expert text classifier. Your task is to classify the following text into one of three categories: "Anti-Vaccine", "Neutral", or "Pro-Vaccine". Here are the definitions of each category:
Anti-Vaccine: means someone who has a negative opinion about the vaccine and its sidelines and opposes its use It does not recommend,
Neutral: means the sentence is news or quotation, the user's position is neutral.
Pro-Vaccine:pro-vaccine classes means someone who encourages and supports the vaccine and vaccination and its sidelines
```
{comment}
```
Your response should be one of the following labels: Anti-Vaccine, Neutral, Pro-Vaccine.
"""

In [ ]:
text = df['Link_removed'].to_list()

In [ ]:
awnser = []

In [ ]:
stopIter = len(text)
counter = 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
while True:
  try:
    if counter >= stopIter:
      break
    res = get_completion(prompt_template.format(comment=text[counter]))
    # append response to text file
    f = open("/content/drive/MyDrive/stance.txt", "a")
    f.write('\n')
    f.write(str(counter)+' : ' + res)
    awnser.append(res)
    f.write(',')
    f.close()
    counter +=1
  except Exception  as e:
    print(e)
    # for limit rate wait for 20 seconds
    sleep(20)

In [ ]:
exist = pd.read_csv(r'/content/stance.txt',header=None)
index = exist[0].str.split(':',expand=True)
index[0] = index[0].astype(int)
index.sort_values(by=0,inplace=True)

In [ ]:
exist

In [ ]:
label=[]
for i in range(len(exist)):
  label.append(exist.iloc[i][0].split(':')[1])

In [ ]:
label=pd.DataFrame(label,columns=['label'])

In [ ]:
label

In [ ]:
label['label'] = label['label'].astype(str)

label['label'] = label['label'].str.strip()

unique_labels = label['label'].unique()
print("Unique labels:", unique_labels)

label_rows = label[label['label'] == 'Label']

print(label_rows)


In [ ]:
label

In [ ]:
df2=pd.concat([label,exist],axis=1)

In [ ]:
df2

In [ ]:
p_tests = []
for i in range(len(df2)):
    label = df2.iloc[i]['label'].strip().lower() 
    if label == 'anti-vaccine':
        p_tests.append(0)
    elif label == 'pro-vaccine':
        p_tests.append(1)
    elif label == 'neutral':
        p_tests.append(2)


In [ ]:
p_tests